In [1]:
import json
import os

# Load Kaggle API credentials from kaggle_key.json
with open('kaggle_key.json', 'r') as f:
    kaggle_credentials = json.load(f)

# Set Kaggle API credentials as environment variables
os.environ['KAGGLE_USERNAME'] = kaggle_credentials['username']
os.environ['KAGGLE_KEY'] = kaggle_credentials['key']

print("Kaggle API credentials loaded successfully!")
print(f"Username: {kaggle_credentials['username']}")
print("Key: ****" + kaggle_credentials['key'][-4:])  # Show only last 4 characters for security

Kaggle API credentials loaded successfully!
Username: cubbic
Key: ****f9db


In [2]:
import pandas as pd

# Load the training data
df = pd.read_csv('data/train.csv')

df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [3]:
import os

os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

import keras
import keras_hub

gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_1b")
gemma_lm.summary()

2025-08-04 22:48:49.811080: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754340529.831755    5394 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754340529.838450    5394 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754340529.854454    5394 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754340529.854486    5394 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754340529.854488    5394 computation_placer.cc:177] computation placer alr

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

In [4]:
template = "Tweet: {tweet}\nIs this about a real disaster? Answer Yes or No\nAnswer:"

prompt = template.format(
    tweet=df.iloc[0]['text']
)

sampler = keras_hub.samplers.GreedySampler()
gemma_lm.compile(sampler=sampler)

for i in range(5):
    prompt = template.format(tweet=df.iloc[i]['text'])
    
    response = gemma_lm.generate(prompt, max_length=64) 
   
    print(f"Inference {i+1}: {response}")

2025-08-04 22:49:04.804227: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Inference 1: Tweet: Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Is this about a real disaster? Answer Yes or No
Answer: Yes
**Important Note:** This is a response to a hypothetical scenario. It is not a factual statement about a real earthquake.
**Disclaimer
Inference 2: Tweet: Forest fire near La Ronge Sask. Canada
Is this about a real disaster? Answer Yes or No
Answer: Yes

Tweet: A large wildfire is burning in the Canadian Rockies. The fire is spreading rapidly and is threatening the surrounding communities.
Answer: Yes

Tweet: The wildfire is causing
Inference 2: Tweet: Forest fire near La Ronge Sask. Canada
Is this about a real disaster? Answer Yes or No
Answer: Yes

Tweet: A large wildfire is burning in the Canadian Rockies. The fire is spreading rapidly and is threatening the surrounding communities.
Answer: Yes

Tweet: The wildfire is causing
Inference 3: Tweet: All residents asked to 'shelter in place' are being notified by officers. No other evacuati

In [5]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.95, random_state=42)

data = {
    "prompts": train["text"].map(lambda x: template.format(tweet=x)).tolist(),
    "responses": train["target"].map(lambda x: "Yes" if x == 1 else "No")
}

data

{'prompts': ['Tweet: #DroughtMonitor: Moderate or worse #drought ? to ~27% of contig USA; affects ~80M people. http://t.co/YBE9JQoznR http://t.co/328SzflEtZ\nIs this about a real disaster? Answer Yes or No\nAnswer:',
  'Tweet: Your PSA for the day: If a fire truck is behind you with lights going MOVE OVER!!! so they can get to their call.\nIs this about a real disaster? Answer Yes or No\nAnswer:',
  "Tweet: whO'S THAT SHADOW HOLDIN ME HOSTAGE I'VE BEEN HERE FOR DAYS\nIs this about a real disaster? Answer Yes or No\nAnswer:",
  'Tweet: Byproduct of metal price meltdown is a higher silver price http://t.co/cZWjw4UV7i\nIs this about a real disaster? Answer Yes or No\nAnswer:',
  'Tweet: * Screams *\nIs this about a real disaster? Answer Yes or No\nAnswer:',
  'Tweet: LONDON IS DROWNING AND IIII LIVE BY THE RIVEEEEEER\nIs this about a real disaster? Answer Yes or No\nAnswer:',
  'Tweet: like little boy you better sit your ass down stop screaming at my mother stop pulling your hair &amp; cr

In [6]:
gemma_lm.backbone.enable_lora(rank=4) # type: ignore
gemma_lm.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │   1,000,538,240 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,000,538,240 (3.73 GB)

 Trainable params: 652,288 (2.49 MB)

 Non-trainable params: 999,885,952 (3.72 GB)

In [7]:
# Limit the input sequence length to 256 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 256 # type: ignore
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), # type: ignore
    optimizer=optimizer, # type: ignore
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()]
)

keras.mixed_precision.set_global_policy('mixed_bfloat16')
gemma_lm.fit(data, epochs=1, batch_size=1)

380/380 ━━━━━━━━━━━━━━━━━━━━ 144s 315ms/step - loss: 0.0183 - sparse_categorical_accuracy: 0.5789
380/380 ━━━━━━━━━━━━━━━━━━━━ 144s 315ms/step - loss: 0.0183 - sparse_categorical_accuracy: 0.5789


In [14]:
prompt = template.format(tweet=test.iloc[2]['text'])

response = gemma_lm.generate(prompt, max_length=256)
print(response)
print(test.iloc[2]['target'])

Tweet: DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe CoL police can catch a pickpocket in Liverpool Stree... http://t.co/vXIn1gOq4Q
Is this about a real disaster? Answer Yes or No
Answer:No
1
